# 02 — Mots de passe et Argon2

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- expliquer pourquoi un hash simple (SHA-256) ne suffit pas pour les mots de passe ;
- utiliser `argon2-cffi` pour hacher et vérifier des mots de passe ;
- configurer les paramètres d'Argon2 (temps, mémoire, parallélisme) ;
- découvrir les alternatives (`bcrypt`, `scrypt`, `hashlib.pbkdf2_hmac`) ;
- appliquer les bonnes pratiques OWASP pour le stockage des mots de passe.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le hachage avec `hashlib` et HMAC (notebook précédent) ;
- la différence entre hash et chiffrement ;
- les types `bytes` et `str`.

## Plan

1. Pourquoi pas SHA-256 ?
2. Fonctions de hachage de mots de passe — principes
3. Argon2 — le standard actuel
4. `argon2-cffi` en pratique
5. Configurer les paramètres
6. Alternatives : bcrypt, scrypt, PBKDF2
7. Bonnes pratiques OWASP
8. Synthèse
9. Exercices
10. Ressources

---

## 1. Pourquoi pas SHA-256 ?

SHA-256 est **trop rapide** pour les mots de passe. Un GPU moderne calcule **~10 milliards** de SHA-256 par seconde.

| Longueur | Alphabet | Combinaisons | Temps SHA-256 (GPU) |
|---|---|---|---|
| 6 caractères | a-z | 309 M | **0.03 secondes** |
| 8 caractères | a-z, 0-9 | 2.8 T | **5 minutes** |
| 8 caractères | a-z, A-Z, 0-9 | 218 T | **6 heures** |

Même avec un sel, un attaquant qui a volé la base peut tester des milliards de candidats par seconde.

### Trois attaques contre les hashes rapides

| Attaque | Description | Protection |
|---|---|---|
| **Brute force** | Tester toutes les combinaisons | Hash lent |
| **Dictionnaire** | Tester les mots courants | Hash lent + sel |
| **Rainbow table** | Table précalculée hash → mot | Sel unique par mot de passe |

---

## 2. Fonctions de hachage de mots de passe — principes

Une bonne fonction de hachage de mot de passe doit être :

1. **Lente** (configurable) — pour rendre le brute force impraticable ;
2. **Salée** — sel aléatoire unique par mot de passe ;
3. **Memory-hard** — utiliser beaucoup de mémoire pour empêcher les attaques GPU ;
4. **Résistante au parallélisme** — limiter le gain des architectures parallèles.

| Fonction | Année | Sel | Lenteur configurable | Memory-hard | Recommandation |
|---|---|---|---|---|---|
| MD5/SHA | — | Non | Non | Non | **Interdit** |
| PBKDF2 | 2000 | Oui | Oui (itérations) | Non | Acceptable |
| bcrypt | 1999 | Oui | Oui (cost factor) | Partiellement | Bon |
| scrypt | 2009 | Oui | Oui | Oui | Bon |
| **Argon2** | 2015 | Oui | Oui | **Oui** | **Recommandé** |

---

## 3. Argon2 — le standard actuel

Argon2 a gagné le **Password Hashing Competition** (PHC) en 2015. C'est la recommandation actuelle d'OWASP.

| Variante | Description |
|---|---|
| **Argon2id** | Hybride (résiste aux side-channel ET au brute force) — **recommandé** |
| Argon2i | Résiste aux side-channel attacks |
| Argon2d | Résiste au brute force GPU |

Argon2 prend trois paramètres :
- **time_cost** : nombre de passes (itérations) ;
- **memory_cost** : mémoire utilisée en Ko ;
- **parallelism** : nombre de threads.

---

## 4. `argon2-cffi` en pratique

> **Installation :** `pip install argon2-cffi`

In [ ]:
try:
    from argon2 import PasswordHasher

    ph = PasswordHasher()
    print("argon2-cffi installé")
except ImportError:
    print("argon2-cffi non installé — pip install argon2-cffi")

### Hacher un mot de passe

In [ ]:
try:
    from argon2 import PasswordHasher

    ph = PasswordHasher()
    hash_stocke = ph.hash("MonMotDePasse123!")
    print(f"Hash : {hash_stocke}")
    print(f"Longueur : {len(hash_stocke)} caractères")
except ImportError:
    print("argon2-cffi non installé")

Le hash Argon2 contient **tout** ce qu'il faut pour la vérification :
```
$argon2id$v=19$m=65536,t=3,p=4$<sel_base64>$<hash_base64>
```

| Champ | Signification |
|---|---|
| `argon2id` | Variante utilisée |
| `v=19` | Version du format |
| `m=65536` | Mémoire (64 Mo) |
| `t=3` | 3 itérations |
| `p=4` | 4 threads |
| `<sel>` | Sel aléatoire (base64) |
| `<hash>` | Hash résultant (base64) |

### Vérifier un mot de passe

In [ ]:
try:
    from argon2 import PasswordHasher
    from argon2.exceptions import VerifyMismatchError

    ph = PasswordHasher()
    hash_stocke = ph.hash("MonMotDePasse123!")

    # Vérification correcte
    try:
        ph.verify(hash_stocke, "MonMotDePasse123!")
        print("Mot de passe correct")
    except VerifyMismatchError:
        print("Mot de passe incorrect")

    # Vérification incorrecte
    try:
        ph.verify(hash_stocke, "MauvaisMotDePasse")
        print("Mot de passe correct")
    except VerifyMismatchError:
        print("Mot de passe incorrect")

except ImportError:
    print("argon2-cffi non installé")

### Re-hashage automatique

In [ ]:
try:
    from argon2 import PasswordHasher

    ph = PasswordHasher()
    hash_stocke = ph.hash("test")

    # check_needs_rehash détecte si les paramètres ont changé
    print(f"Besoin de re-hasher : {ph.check_needs_rehash(hash_stocke)}")

    # Si on change les paramètres...
    ph2 = PasswordHasher(time_cost=5, memory_cost=131072)
    print(f"Besoin de re-hasher : {ph2.check_needs_rehash(hash_stocke)}")
    # True — le hash a été créé avec des paramètres différents

except ImportError:
    print("argon2-cffi non installé")

Le re-hashage permet de **migrer progressivement** vers des paramètres plus forts : à chaque connexion réussie, si `check_needs_rehash` retourne `True`, on re-hache le mot de passe avec les nouveaux paramètres.

---

## 5. Configurer les paramètres

Les paramètres par défaut de `argon2-cffi` sont déjà bons, mais vous pouvez les ajuster selon votre hardware.

In [ ]:
try:
    from argon2 import PasswordHasher, Type

    # Paramètres par défaut
    ph = PasswordHasher()
    print(f"time_cost    : {ph.time_cost}")
    print(f"memory_cost  : {ph.memory_cost} Ko ({ph.memory_cost // 1024} Mo)")
    print(f"parallelism  : {ph.parallelism}")
    print(f"hash_len     : {ph.hash_len}")
    print(f"salt_len     : {ph.salt_len}")
    print(f"type         : {ph.type}")

except ImportError:
    print("argon2-cffi non installé")

### Calibrer les paramètres

In [ ]:
try:
    import time
    from argon2 import PasswordHasher

    # Objectif : un hash doit prendre entre 200 ms et 1 s
    for mem_mo in [32, 64, 128, 256]:
        for t in [1, 2, 3]:
            ph = PasswordHasher(time_cost=t, memory_cost=mem_mo * 1024)
            start = time.perf_counter()
            ph.hash("test")
            duree = time.perf_counter() - start
            print(f"mem={mem_mo:3d}Mo  t={t}  → {duree*1000:.0f} ms")

except ImportError:
    print("argon2-cffi non installé")

### Recommandations OWASP (2024)

| Paramètre | Minimum | Recommandé |
|---|---|---|
| Mémoire | 19 Mo | 64 Mo |
| Itérations | 2 | 3 |
| Parallélisme | 1 | 4 |
| Durée cible | 200 ms | 500 ms – 1 s |

---

## 6. Alternatives : bcrypt, scrypt, PBKDF2

### `bcrypt`

In [ ]:
try:
    import bcrypt

    mdp = b"MonMotDePasse123!"
    sel = bcrypt.gensalt(rounds=12)  # 2^12 itérations
    hash_bcrypt = bcrypt.hashpw(mdp, sel)
    print(f"bcrypt : {hash_bcrypt.decode()}")

    # Vérification
    print(f"Correct : {bcrypt.checkpw(mdp, hash_bcrypt)}")
    print(f"Faux    : {bcrypt.checkpw(b'mauvais', hash_bcrypt)}")

except ImportError:
    print("bcrypt non installé — pip install bcrypt")

### `hashlib.scrypt` (bibliothèque standard)

In [ ]:
import hashlib
import os

mdp = b"MonMotDePasse123!"
sel = os.urandom(16)

# scrypt est memory-hard comme Argon2
hash_scrypt = hashlib.scrypt(mdp, salt=sel, n=16384, r=8, p=1, dklen=32)
print(f"scrypt : {hash_scrypt.hex()}")

### `hashlib.pbkdf2_hmac` (bibliothèque standard)

In [ ]:
sel = os.urandom(16)
hash_pbkdf2 = hashlib.pbkdf2_hmac("sha256", mdp, sel, iterations=600_000)
print(f"PBKDF2 : {hash_pbkdf2.hex()}")

### Comparaison

| Critère | Argon2 | bcrypt | scrypt | PBKDF2 |
|---|---|---|---|---|
| Memory-hard | Oui | Non | Oui | Non |
| Standard | PHC 2015 | De facto | — | NIST SP 800-132 |
| Résistance GPU | Excellente | Bonne | Bonne | Faible |
| Bibliothèque standard | Non | Non | Oui | Oui |
| Recommandation OWASP | **1er choix** | 2e choix | 3e choix | Acceptable |

---

## 7. Bonnes pratiques OWASP

1. **Utilisez Argon2id** comme premier choix.
2. **Générez un sel unique** par mot de passe (les bibliothèques le font automatiquement).
3. **Ne limitez pas** la longueur du mot de passe (au-delà du raisonnable, 128 caractères max).
4. **Ne tronquez pas** le mot de passe avant hachage (sauf limite anti-DoS).
5. **Stockez le hash complet** (paramètres + sel + hash) — format PHC.
6. **Migrez progressivement** avec `check_needs_rehash`.
7. **Ne loggez jamais** un mot de passe en clair.
8. **Peppering** (optionnel) : HMAC avec un secret côté serveur avant Argon2.

### Pattern de migration

In [ ]:
try:
    from argon2 import PasswordHasher
    from argon2.exceptions import VerifyMismatchError

    # Simuler une base de données
    db = {}

    def creer_compte(username, password):
        ph = PasswordHasher()
        db[username] = ph.hash(password)

    def authentifier(username, password):
        if username not in db:
            return False
        ph = PasswordHasher()  # avec les paramètres ACTUELS
        try:
            ph.verify(db[username], password)
        except VerifyMismatchError:
            return False

        # Migration transparente si les paramètres ont changé
        if ph.check_needs_rehash(db[username]):
            db[username] = ph.hash(password)
            print(f"  [migration] re-hashé pour {username}")
        return True

    creer_compte("alice", "SuperSecret42!")
    print(f"Auth réussie : {authentifier('alice', 'SuperSecret42!')}")
    print(f"Auth échouée : {authentifier('alice', 'mauvais')}")

except ImportError:
    print("argon2-cffi non installé")

---

## 8. Synthèse

| Outil | Usage |
|---|---|
| `argon2-cffi` | Hachage de mots de passe (recommandé) |
| `bcrypt` | Alternative éprouvée |
| `hashlib.scrypt` | Memory-hard, stdlib |
| `hashlib.pbkdf2_hmac` | Acceptable, stdlib |

**Règles à retenir :**
- **Jamais** SHA-256/MD5 pour les mots de passe.
- Argon2id est le standard actuel (OWASP, PHC).
- Le sel est généré automatiquement par les bibliothèques — ne le faites pas manuellement.
- Visez 200 ms – 1 s par hash (calibrez sur votre serveur).
- Migrez progressivement avec `check_needs_rehash`.

---

## 9. Exercices

### Exercice 1 — Hacher et vérifier *(facile)*

Utilisez `argon2-cffi` pour :
1. Hacher le mot de passe `"Python3.14!"`
2. Vérifier que le hash correspond
3. Vérifier qu'un mauvais mot de passe est rejeté

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Passwords_argon2", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from argon2 import PasswordHasher
from argon2.exceptions import VerifyMismatchError

ph = PasswordHasher()
hash_stocke = ph.hash("Python3.14!")

# Vérification correcte
try:
    ph.verify(hash_stocke, "Python3.14!")
    print("OK : mot de passe correct")
except VerifyMismatchError:
    print("ERREUR")

# Vérification incorrecte
try:
    ph.verify(hash_stocke, "Python2.7")
    print("ERREUR : ne devrait pas passer")
except VerifyMismatchError:
    print("OK : mot de passe rejeté")
```

</details>

### Exercice 2 — Calibration automatique *(moyen)*

Écrire une fonction `calibrer_argon2(duree_cible_ms: int = 500) -> dict` qui teste différentes combinaisons de `time_cost` et `memory_cost` et retourne la configuration qui s'approche le plus de la durée cible.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Passwords_argon2", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
from argon2 import PasswordHasher

def calibrer_argon2(duree_cible_ms: int = 500) -> dict:
    meilleur = None
    meilleur_ecart = float("inf")

    for mem_mo in [16, 32, 64, 128, 256]:
        for t in [1, 2, 3, 4, 5]:
            ph = PasswordHasher(time_cost=t, memory_cost=mem_mo * 1024)
            start = time.perf_counter()
            ph.hash("calibration_test")
            duree_ms = (time.perf_counter() - start) * 1000

            ecart = abs(duree_ms - duree_cible_ms)
            if ecart < meilleur_ecart:
                meilleur_ecart = ecart
                meilleur = {
                    "time_cost": t,
                    "memory_cost": mem_mo * 1024,
                    "duree_mesuree_ms": round(duree_ms),
                }

    return meilleur

config = calibrer_argon2(500)
print(f"Configuration optimale : {config}")
```

</details>

### Exercice 3 — Service d'authentification complet *(moyen)*

Implémenter une classe `AuthService` avec :
- `creer_compte(username, password)` — crée un compte ;
- `authentifier(username, password) -> bool` — vérifie le mot de passe ;
- `changer_mdp(username, ancien, nouveau)` — change le mot de passe ;
- migration automatique des hashes obsolètes.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Passwords_argon2", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from argon2 import PasswordHasher
from argon2.exceptions import VerifyMismatchError

class AuthService:
    def __init__(self, **kwargs):
        self.ph = PasswordHasher(**kwargs)
        self._db = {}

    def creer_compte(self, username: str, password: str) -> None:
        if username in self._db:
            raise ValueError(f"Compte {username!r} existe déjà")
        self._db[username] = self.ph.hash(password)

    def authentifier(self, username: str, password: str) -> bool:
        if username not in self._db:
            # Éviter le timing leak : hasher quand même
            self.ph.hash(password)
            return False
        try:
            self.ph.verify(self._db[username], password)
        except VerifyMismatchError:
            return False
        if self.ph.check_needs_rehash(self._db[username]):
            self._db[username] = self.ph.hash(password)
        return True

    def changer_mdp(self, username: str, ancien: str, nouveau: str) -> bool:
        if not self.authentifier(username, ancien):
            return False
        self._db[username] = self.ph.hash(nouveau)
        return True

auth = AuthService()
auth.creer_compte("alice", "Secret42!")
print(f"Auth OK : {auth.authentifier('alice', 'Secret42!')}")
print(f"Auth KO : {auth.authentifier('alice', 'mauvais')}")
print(f"Change  : {auth.changer_mdp('alice', 'Secret42!', 'NouveauMdp!')}")
print(f"Auth new: {auth.authentifier('alice', 'NouveauMdp!')}")
```

</details>

### Exercice 4 — Comparer les performances des algorithmes *(difficile)*

Écrire un benchmark qui compare le temps de hachage de Argon2, bcrypt, scrypt et PBKDF2 avec des paramètres visant une durée cible de ~500 ms chacun. Présentez les résultats dans un tableau.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Passwords_argon2", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
import hashlib
import os

results = []

# Argon2
try:
    from argon2 import PasswordHasher
    ph = PasswordHasher(time_cost=3, memory_cost=65536)
    t0 = time.perf_counter()
    ph.hash("benchmark")
    dt = time.perf_counter() - t0
    results.append(("Argon2id", f"{dt*1000:.0f} ms", "memory-hard"))
except ImportError:
    results.append(("Argon2id", "non installé", ""))

# bcrypt
try:
    import bcrypt
    t0 = time.perf_counter()
    bcrypt.hashpw(b"benchmark", bcrypt.gensalt(rounds=13))
    dt = time.perf_counter() - t0
    results.append(("bcrypt", f"{dt*1000:.0f} ms", "CPU-hard"))
except ImportError:
    results.append(("bcrypt", "non installé", ""))

# scrypt
sel = os.urandom(16)
t0 = time.perf_counter()
hashlib.scrypt(b"benchmark", salt=sel, n=32768, r=8, p=1)
dt = time.perf_counter() - t0
results.append(("scrypt", f"{dt*1000:.0f} ms", "memory-hard"))

# PBKDF2
t0 = time.perf_counter()
hashlib.pbkdf2_hmac("sha256", b"benchmark", sel, iterations=600_000)
dt = time.perf_counter() - t0
results.append(("PBKDF2", f"{dt*1000:.0f} ms", "CPU-only"))

print(f"{'Algorithme':<12s} {'Durée':>12s} {'Type':>15s}")
print("-" * 42)
for nom, duree, type_ in results:
    print(f"{nom:<12s} {duree:>12s} {type_:>15s}")
```

</details>

---

## 10. Ressources

- [`argon2-cffi` — documentation](https://argon2-cffi.readthedocs.io/)
- [OWASP Password Storage Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html)
- [PHC — Password Hashing Competition](https://www.password-hashing.net/)
- [Argon2 — RFC 9106](https://www.rfc-editor.org/rfc/rfc9106)